# NF Generalize Nick Data: SSCD + Physics Quickcheck

This notebook is the SSCD version of the `nf_generalize_nick_data` generalizability check.

Paper-style score:

```text
s_j = max_i SSCD_cosine(generated_j, train_i)
GL(tau) = 1 - mean_j[s_j > tau]
```

The paper uses `tau = 0.6`. We also plot other fixed thresholds to see whether the conclusion is threshold-sensitive.

Important: SSCD/GL is a **copy audit**, not a physics-fidelity metric. A sample can be far from every training image because it is genuinely novel, or because it is bad/off-manifold. That is why this notebook also keeps one-point statistics, P(k), and real-vs-generated image panels.


## Run Commands

First run the offline SSCD job. This avoids killing the notebook kernel while embedding all training slices.

```bash
cd /home/jiamingp/diffusion_models_repo
python scripts/prepare_nf_generalize_nick_data_configs.py --project-dir "$PWD" --check-only
sbatch -A huterer0 scripts/slurm/analyze_nf_generalize_sscd_full_nn.sbatch
```

If the SSCD checkpoint is missing:

```bash
mkdir -p ~/.cache/torch/hub
curl -L -o ~/.cache/torch/hub/sscd_disc_mixup.torchscript.pt   https://dl.fbaipublicfiles.com/sscd-copy-detection/sscd_disc_mixup.torchscript.pt
```

The offline job writes:

```text
results/nf_generalize_nick_data/tables/nf_generalize_nick_data_sscd_full_nn_metrics.csv
results/nf_generalize_nick_data/quickcheck/nf_generalize_nick_data_sscd_full_nn_paper_style_gl_curves.png
results/nf_generalize_nick_data/quickcheck/nf_generalize_nick_data_sscd_full_nn_similarity_curves.png
results/nf_generalize_nick_data/quickcheck/nf_generalize_nick_data_sscd_full_nn_copy_fraction_curves.png
```


In [ ]:
from __future__ import annotations

import json
import math
import os
import sys
from pathlib import Path
from typing import Any

import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
import numpy as np
import pandas as pd
from IPython.display import Image, display

PROJECT_CANDIDATES = [
    Path.cwd(),
    Path('/home/jiamingp/diffusion_models_repo'),
    Path('/Users/apple/AI/Diffusion_model'),
]
PROJECT_DIR = next((p for p in PROJECT_CANDIDATES if (p / 'simdiff_eval').exists()), PROJECT_CANDIDATES[0])
if str(PROJECT_DIR) not in sys.path:
    sys.path.insert(0, str(PROJECT_DIR))

from simdiff_eval.io import as_nchw, load_real_from_config
from simdiff_eval.metrics import batch_power_spectra, field_histogram, power_spectrum_summary

SWEEP_NAME = 'nf_generalize_nick_data'
MANIFEST_PATH = PROJECT_DIR / 'local' / SWEEP_NAME / 'manifest.json'
CONFIG_DIR = PROJECT_DIR / 'local' / SWEEP_NAME / 'configs'
SAMPLE_ROOT = PROJECT_DIR / 'results' / SWEEP_NAME / 'samples'
OUTPUT_DIR = PROJECT_DIR / 'results' / SWEEP_NAME / 'quickcheck'
TABLE_DIR = PROJECT_DIR / 'results' / SWEEP_NAME / 'tables'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
TABLE_DIR.mkdir(parents=True, exist_ok=True)

SEED = int(os.environ.get('NF_GEN_NICK_SEED', 123))
SAMPLE_LABEL = os.environ.get('NF_GEN_NICK_SAMPLE_LABEL', 'raw_train_full')
MAX_GENERATED = int(os.environ.get('NF_GEN_NICK_MAX_GENERATED', 512))
MAX_REAL_HIST = int(os.environ.get('NF_GEN_NICK_MAX_REAL_HIST', 512))
MAX_REAL_PK = int(os.environ.get('NF_GEN_NICK_MAX_REAL_PK', 512))
PK_NBINS = int(os.environ.get('NF_GEN_NICK_PK_NBINS', 30))
RUN_PHYSICS = os.environ.get('NF_GEN_NICK_RUN_PHYSICS', '1') == '1'

SSCD_METRICS_PATH = TABLE_DIR / 'nf_generalize_nick_data_sscd_full_nn_metrics.csv'
SSCD_GL_FIG = OUTPUT_DIR / 'nf_generalize_nick_data_sscd_full_nn_paper_style_gl_curves.png'
SSCD_SIM_FIG = OUTPUT_DIR / 'nf_generalize_nick_data_sscd_full_nn_similarity_curves.png'
SSCD_COPY_FIG = OUTPUT_DIR / 'nf_generalize_nick_data_sscd_full_nn_copy_fraction_curves.png'

print('PROJECT_DIR =', PROJECT_DIR)
print('MANIFEST_PATH =', MANIFEST_PATH)
print('SAMPLE_ROOT =', SAMPLE_ROOT)
print('RUN_PHYSICS =', RUN_PHYSICS)


In [ ]:
def dataset_size(row: dict[str, Any]) -> int:
    return int(row.get('dataset_size', row.get('actual_2d', row.get('target_2d'))))


def config_path_for(row: dict[str, Any]) -> Path:
    raw = row.get('config') or f"local/{SWEEP_NAME}/configs/{row['run_name']}.yaml"
    path = Path(raw)
    return path if path.is_absolute() else PROJECT_DIR / path


def sample_path_for(row: dict[str, Any]) -> Path:
    return SAMPLE_ROOT / f"{row['run_name']}_seed{SEED}_{SAMPLE_LABEL}.npz"


def load_npz_array(path: Path) -> np.ndarray:
    with np.load(path) as data:
        if 'samples' in data:
            arr = data['samples']
        elif 'arr_0' in data:
            arr = data['arr_0']
        else:
            arr = data[data.files[0]]
    return as_nchw(np.asarray(arr, dtype=np.float32))


def evenly_limit(arr: np.ndarray, limit: int | None) -> np.ndarray:
    arr = np.asarray(arr)
    if limit is None or len(arr) <= limit:
        return arr.copy()
    idx = np.linspace(0, len(arr) - 1, limit, dtype=int)
    return arr[idx].copy()


def raw_sim_cap_for(row: dict[str, Any], slice_cap: int | None) -> int | None:
    if slice_cap is None:
        return None
    slices_per_sim = int(row.get('slices_per_sim') or max(1, 128 // int(row.get('zthin', 8))))
    cap = int(math.ceil(int(slice_cap) / slices_per_sim))
    total = row.get('n_train_simulations') or row.get('n_samples_simulations')
    if total is not None:
        cap = min(cap, int(total))
    return max(1, cap)


def load_rows() -> list[dict[str, Any]]:
    with MANIFEST_PATH.open() as f:
        rows = json.load(f)
    return sorted(rows, key=dataset_size)

rows = load_rows()
status_rows = []
for row in rows:
    sample_path = sample_path_for(row)
    status_rows.append({
        'run_name': row['run_name'],
        'dataset_tag': row.get('dataset_tag'),
        'dataset_size': dataset_size(row),
        'sample_exists': sample_path.exists(),
        'sample_path': str(sample_path),
    })
status_df = pd.DataFrame(status_rows)
display(status_df)
print('samples found:', int(status_df['sample_exists'].sum()), '/', len(status_df))


## SSCD Generalizability Results

Definitions:

```text
s_gen[j] = max_i SSCD_cosine(generated_j, train_i)
copy_fraction(tau) = mean_j[s_gen[j] > tau]
GL(tau) = 1 - copy_fraction(tau)
```

For the Zhang et al. Figure 2-style metric, use the `tau=0.6` curve. If the curve changes strongly with `tau`, then the conclusion is not robust to the copy threshold.

How to read this with your image panel: larger `N` samples looking more realistic can increase nearest-training similarity because realistic samples live on the same CAMELS manifold. SSCD asks whether they are close to a particular training slice, but it still must be interpreted with P(k), one-point stats, and visual quality.


In [ ]:
sscd_df = pd.DataFrame()
if SSCD_METRICS_PATH.exists():
    sscd_df = pd.read_csv(SSCD_METRICS_PATH).sort_values('dataset_size')
    cols = [
        'dataset_tag', 'dataset_size', 'n_train_ref', 'n_val_real', 'n_generated',
        'gen_nn_median', 'gen_nn_q99', 'val_nn_median', 'threshold_q99',
        'gen_gl_fixed_0p6', 'gen_copy_fraction_fixed_0p6',
    ]
    cols = [c for c in cols if c in sscd_df.columns]
    display(sscd_df[cols])
    print('loaded', SSCD_METRICS_PATH)
else:
    print('Missing SSCD metrics:', SSCD_METRICS_PATH)
    print('Run: sbatch -A huterer0 scripts/slurm/analyze_nf_generalize_sscd_full_nn.sbatch')

for fig_path in [SSCD_GL_FIG, SSCD_SIM_FIG, SSCD_COPY_FIG]:
    if fig_path.exists():
        print(fig_path)
        display(Image(filename=str(fig_path)))
    else:
        print('missing figure:', fig_path)


## Physics Diagnostics: One-Point, P(k), Images

These are deliberately kept beside SSCD. If SSCD says “not copied” but P(k)/one-point/images are bad, then the model is not a good generalizer; it is just not a near-copy detector hit.

This section loads capped real slices for speed. Increase `NF_GEN_NICK_MAX_REAL_HIST` and `NF_GEN_NICK_MAX_REAL_PK` for lower-noise final plots.


In [ ]:
loaded: dict[str, dict[str, Any]] = {}
real_cache: dict[tuple[str, int], np.ndarray] = {}

if RUN_PHYSICS:
    real_slice_cap = max(MAX_REAL_HIST, MAX_REAL_PK)
    for row in rows:
        sample_path = sample_path_for(row)
        if not sample_path.exists():
            continue
        generated = evenly_limit(load_npz_array(sample_path), MAX_GENERATED)
        raw_cap = raw_sim_cap_for(row, real_slice_cap)
        key = (row['run_name'], raw_cap or -1)
        if key not in real_cache:
            real_cache[key] = evenly_limit(
                as_nchw(load_real_from_config(config_path_for(row), max_raw_samples=raw_cap)),
                real_slice_cap,
            )
        loaded[row['run_name']] = {
            'spec': row,
            'real': real_cache[key],
            'generated': generated,
            'sample_path': sample_path,
        }

loaded_rows = []
for run_name, bundle in loaded.items():
    row = bundle['spec']
    loaded_rows.append({
        'run_name': run_name,
        'dataset_tag': row.get('dataset_tag'),
        'dataset_size': dataset_size(row),
        'n_real_loaded': len(bundle['real']),
        'n_generated': len(bundle['generated']),
    })
loaded_df = pd.DataFrame(loaded_rows).sort_values('dataset_size') if loaded_rows else pd.DataFrame()
display(loaded_df)
print('loaded physics rows:', len(loaded))


In [ ]:
physics_records = []
if loaded:
    for run_name, bundle in sorted(loaded.items(), key=lambda kv: dataset_size(kv[1]['spec'])):
        row = bundle['spec']
        real = evenly_limit(bundle['real'], MAX_REAL_HIST)
        generated = evenly_limit(bundle['generated'], MAX_GENERATED)
        real_hist = field_histogram(real)
        gen_hist = field_histogram(generated)
        edges = np.asarray(real_hist['bin_edges'])
        bin_width = float(np.mean(np.diff(edges)))
        hist_l1 = float(np.sum(np.abs(np.asarray(real_hist['hist']) - np.asarray(gen_hist['hist']))) * bin_width)
        pk_summary = power_spectrum_summary(
            evenly_limit(bundle['real'], MAX_REAL_PK),
            generated,
            nbins=PK_NBINS,
        )
        physics_records.append({
            'run_name': run_name,
            'dataset_tag': row.get('dataset_tag'),
            'dataset_size': dataset_size(row),
            'n_real_hist': len(real),
            'n_generated': len(generated),
            'hist_l1': hist_l1,
            'real_std': real_hist['std'],
            'generated_std': gen_hist['std'],
            'std_ratio': gen_hist['std'] / max(real_hist['std'], 1e-30),
            **pk_summary,
        })

physics_df = pd.DataFrame(physics_records).sort_values('dataset_size') if physics_records else pd.DataFrame()
if len(physics_df):
    display(physics_df)
    out = TABLE_DIR / 'nf_generalize_nick_data_sscd_notebook_physics_metrics.csv'
    physics_df.to_csv(out, index=False)
    print('wrote', out)
else:
    print('No physics records. Set RUN_PHYSICS=True and make sure sample files exist.')


In [ ]:
if len(physics_df):
    x = physics_df['dataset_size'].astype(float)
    fig, axes = plt.subplots(1, 3, figsize=(18, 4.8), sharex=True)
    axes[0].plot(x, physics_df['hist_l1'], 'o-', label='one-point L1')
    axes[0].set_ylabel('one-point histogram L1')
    axes[0].set_title('one-point error')

    axes[1].plot(x, physics_df['pk_log10_mae'], 'o-', label='P(k) log10 MAE')
    axes[1].set_ylabel('P(k) log10 MAE')
    axes[1].set_title('P(k) error')

    axes[2].plot(x, physics_df['std_ratio'], 'o-', label='std ratio')
    axes[2].axhline(1.0, color='0.3', ls=':')
    axes[2].set_ylabel('generated std / real std')
    axes[2].set_title('field variance')

    for ax in axes:
        ax.set_xscale('log', base=2)
        ax.set_xlabel('training dataset size N')
        ax.grid(alpha=0.25)
    fig.suptitle('Physics metrics versus training size')
    fig.tight_layout(rect=(0, 0, 1, 0.92))
    out = OUTPUT_DIR / 'nf_generalize_nick_data_sscd_notebook_physics_curves.png'
    fig.savefig(out, dpi=180, bbox_inches='tight')
    print('wrote', out)
    plt.show()


In [ ]:
if loaded:
    items = sorted(loaded.items(), key=lambda kv: dataset_size(kv[1]['spec']))
    ncols = min(5, len(items))
    nblocks = math.ceil(len(items) / ncols)
    fig, axes = plt.subplots(2 * nblocks, ncols, figsize=(3.0 * ncols, 5.0 * nblocks), squeeze=False)
    for ax in axes.ravel():
        ax.axis('off')

    vals = np.concatenate([
        np.asarray(bundle['real'][:1]).ravel() for _, bundle in items
    ] + [
        np.asarray(bundle['generated'][:1]).ravel() for _, bundle in items
    ])
    vmin, vmax = np.nanpercentile(vals, [1, 99])
    if not np.isfinite(vmin) or not np.isfinite(vmax) or vmin == vmax:
        vmin, vmax = -1, 1

    for idx, (run_name, bundle) in enumerate(items):
        block = idx // ncols
        col = idx % ncols
        row = bundle['spec']
        ax_real = axes[2 * block, col]
        ax_gen = axes[2 * block + 1, col]
        ax_real.imshow(bundle['real'][0, 0], origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
        ax_gen.imshow(bundle['generated'][0, 0], origin='lower', cmap='viridis', vmin=vmin, vmax=vmax)
        ax_real.set_title(f"{row.get('dataset_tag')} real", fontsize=11)
        ax_gen.set_title('generated', fontsize=11)
        ax_real.axis('off')
        ax_gen.axis('off')

    fig.suptitle('real vs generated examples')
    fig.tight_layout(rect=(0, 0, 1, 0.96))
    out = OUTPUT_DIR / 'nf_generalize_nick_data_sscd_notebook_real_vs_generated_examples.png'
    fig.savefig(out, dpi=180, bbox_inches='tight')
    print('wrote', out)
    plt.show()


## Working Interpretation

If SSCD GL decreases with larger `N`, that means generated samples are more often above the SSCD copy threshold. There are two possible readings:

1. **Bad reading:** larger models/training sets are memorizing specific training slices.
2. **Benign reading:** larger-`N` samples are finally realistic enough that SSCD sees them as close to real CAMELS morphology, but not necessarily exact copies.

The distinction is why we compare against held-out real and train-real leave-one-out baselines, and why P(k)/one-point/images remain necessary. If generated SSCD similarity approaches held-out-real similarity while physics metrics improve, that supports better sample quality. If generated samples cross train-real near-duplicate thresholds or nearest-neighbor image pairs look identical, that supports memorization.
